# DRT 반영 이동격차지수: 현재 운행 vs 재배치

경기도 602개 읍면동, 총 326대의 두 시나리오를 비교합니다. 기존 6개 취약점수에 DRT 공급부족 점수를 추가합니다. 같은 정규화·같은 엔트로피 가중치를 사용하며 지수가 낮아질수록 분석상 개선입니다.

현재 차량은 권역별 공동차량의 **가상 귀속대수**입니다. 후보 연결과 균등배분을 포함하며 실제 이용효과를 추정하지 않습니다. 상세 가정은 README.md 및 inputs/route_mapping.json에 있습니다.

이 노트북과 run_comparison.py, inputs 폴더를 함께 유지하세요. pandas, numpy, matplotlib만 필요합니다. Colab에서 실행할 때도 폴더 전체를 먼저 업로드해야 합니다.

In [1]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
base = Path.cwd()
if not (base / 'run_comparison.py').exists():
    base = base / 'DRT_반영_이동격차지수_비교_20261010'
assert (base / 'run_comparison.py').is_file(), '노트북이 있는 폴더에서 실행하세요.'
spec = importlib.util.spec_from_file_location('drt_comparison', base / 'run_comparison.py')
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print('분석 폴더:', base.resolve())

분석 폴더: C:\Users\전가연\OneDrive\Desktop\traffic-contest\DRT_반영_이동격차지수_비교_20261010


## 1. 원본 6지표 재현 및 입력 검증

전국 3,555개 지역의 기존 전처리 결과 스냅샷에서 원본 엔트로피 가중치와 이동격차지수를 다시 계산하여 일치를 확인합니다. 자동차·고령인구 등 원자료 전처리를 새로 수행하는 단계는 아닙니다. 경기도 602개의 기존 지수와 재배치 CSV의 기존 지수도 일치해야 합니다.

In [2]:
national, gg, raw, sim, rules = analysis.load_inputs()
print('전국/경기도 행수:', len(national), len(gg))
print('현재 운행 권역/차량:', len(raw), raw['운행대수'].sum())
print('제공 재배치 차량:', sim['재배치_대수'].sum())
print('제공 시뮬레이션의 현재 귀속대수:', sim['현재_귀속대수'].sum())
print('경기도 기존지표 결측:')
print(gg[analysis.VARS].isna().sum().to_string())

전국/경기도 행수: 3555 602
현재 운행 권역/차량: 61 326
제공 재배치 차량: 326
제공 시뮬레이션의 현재 귀속대수: 300.0
경기도 기존지표 결측:
지형장벽_취약점수       0
자가용미보유_취약점수    29
의료_취약점수         0
버스정류장_취약점수      0
배차간격_취약점수       0
고령인구비율_취약점수    32


## 2. 현재 운행권역을 읍면동에 배분

한 권역의 차량은 대상 읍면동 수로 나누어 균등 귀속합니다. 복수 권역이 연결되면 합산합니다. 61개 권역별 보존 및 총 326대 보존을 검증합니다. 후보와 가정을 유지한 상태에서 계산하며, 연결 없음은 분석상 0대입니다. 300대 현재귀속 열을 그대로 복제하지 않습니다.

In [3]:
current, edges, audit = analysis.map_current(gg, raw, rules)
print(audit[['운행구역ID','시군','운행구역','원본대수','배분합계','연결상태']].to_string(index=False))
print('권역 총량:', audit['원본대수'].sum(), '/ 귀속 합계:', current.sum())

운행구역ID   시군                 운행구역  원본대수  배분합계    연결상태
DRT001  연천군              연천읍·내산리     2   2.0 명칭·관할연결
DRT002  파주시                 운정4동     3   3.0 명칭·관할연결
DRT003  파주시                  광탄면     3   3.0 명칭·관할연결
DRT004  파주시                  탄현면     3   3.0 명칭·관할연결
DRT005  파주시     운정1,2,3지구 및 교하지구    15  15.0      후보
DRT006  양주시                  장흥면     4   4.0 명칭·관할연결
DRT007  양주시          백석읍·광적면·양주역     4   4.0      후보
DRT008  양주시            은현·남면·덕정역     3   3.0      후보
DRT009  양주시 옥정신도시,회천지구,삼숭동 및 덕계역    15  15.0      후보
DRT010  김포시                고촌·풍무    10  10.0 명칭·관할연결
DRT011  고양시                  덕은동     3   3.0      후보
DRT012  고양시                  고봉동     3   3.0 명칭·관할연결
DRT013  고양시                  식사동     4   4.0 명칭·관할연결
DRT014  고양시                  향동동     4   4.0 명칭·관할연결
DRT015 의정부시             민락2·고산지구     4   4.0      후보
DRT016 의정부시                민락1지구     4   4.0      후보
DRT017  하남시              하남 감일지구     5   5.0      후보
DRT018  하남시              하남 위례지구     4   4.0  

## 3. DRT 점수와 공통 엔트로피 가중치

기존 6개 점수는 전국 정규화를 보존합니다. DRT는 두 시나리오 대수를 합친 최소·최대에 대해 `1 - (대수-min)/(max-min)`으로 계산합니다.

경기도 두 시나리오를 세로로 합친 1,204행에서 결측 없는 행으로 엔트로피 가중치를 한 번 계산합니다. 원본과 동일하게 ε=10⁻⁶을 사용하며 최종 합산에는 결측점수 0 대체를 유지합니다. 경기도 외 DRT가 없으므로 전국 7지표 가중치를 계산한 것은 아닙니다.

`P_ij = X'_ij / Σ_i X'_ij`, `E_j = -Σ_i(P_ij ln P_ij) / ln(n)`, `w_j=(1-E_j)/Σ_j(1-E_j)`, `I_i=Σ_j(w_j X_ij)`

In [4]:
result, weights, settings = analysis.calculate(gg, sim, current, edges)
print(settings)
print(weights.to_string(index=False))
print('현재 DRT 점수 범위:', result['현재_DRT공급부족점수'].min(), result['현재_DRT공급부족점수'].max())
print('재배치 DRT 점수 범위:', result['재배치_DRT공급부족점수'].min(), result['재배치_DRT공급부족점수'].max())

{'drt_min': 0.0, 'drt_max': 10.0, 'pooled_rows': 1204, 'entropy_complete_rows': 1140, 'epsilon': 1e-06, 'normalization': '기존6개 전국점수 보존; DRT 두 시나리오 합동 Min-Max', 'weights_population': '경기도 602개 × 2시나리오 중 7지표 완전행', 'missing_policy': '원본과 동일: 가중치 적합시 결측행 제외, 합산시 결측점수 0'}
          지표     엔트로피     정보효용      가중치
   지형장벽_취약점수 0.974112 0.025888 0.135261
 자가용미보유_취약점수 0.999929 0.000071 0.000370
     의료_취약점수 0.937464 0.062536 0.326744
  버스정류장_취약점수 0.999618 0.000382 0.001994
   배차간격_취약점수 0.911069 0.088931 0.464651
 고령인구비율_취약점수 0.987503 0.012497 0.065297
DRT공급부족_취약점수 0.998912 0.001088 0.005682
현재 DRT 점수 범위: 0.0 1.0
재배치 DRT 점수 범위: 0.8 1.0


## 4. 비교 및 산식 독립 확인

공통 가중치와 정규화에서는 `ΔI = -w_DRT × Δ차량/(max-min)`입니다. 차량 총량이 동일하므로 전체 평균 변화는 0이며, 기존 취약지역의 변화와 개선·악화 양쪽을 함께 비교합니다. 이번 결과는 재배치 계획을 실제로 실행한 인과효과가 아닙니다.

In [5]:
summary, city = analysis.summarize(result)
w_drt = weights.set_index('지표').loc[analysis.DRT, '가중치']
expected = -w_drt * result['대수변화_재배치빼기현재'] / (settings['drt_max'] - settings['drt_min'])
assert np.allclose(expected, result['지수변화_재배치빼기현재'], atol=1e-12, rtol=0)
assert abs(result['지수변화_재배치빼기현재'].mean()) < 1e-12
print(summary.to_string(index=False))
print(result['변화판정'].value_counts().to_string())
print(result.nsmallest(20,'기존6지표_순위')[['읍면_key','현재반영_이동격차지수','재배치반영_이동격차지수','지수변화_재배치빼기현재']].to_string(index=False))

        대상 시나리오  읍면동수       대수합계  공급양수_읍면동수     지수평균  지수표준편차_ddof0     지수최대     지수최소  기존지표결측_읍면동수
    경기도 전체   현재   602 326.000000        113 0.109971      0.072577 0.448959 0.032137           32
    경기도 전체  재배치   602 326.000000        313 0.109971      0.072437 0.448390 0.032137           32
기존6지표 상위20   현재    20   9.666667          7 0.348733      0.044480 0.448959 0.292746            0
기존6지표 상위20  재배치    20  33.000000         20 0.348070      0.044302 0.448390 0.292178            0
기존6지표 상위50   현재    50  31.000000         18 0.297420      0.052531 0.448959 0.231137            0
기존6지표 상위50  재배치    50  63.000000         50 0.297057      0.052252 0.448390 0.230569            0
 기존지표 결측없음   현재   570 286.000000         98 0.110704      0.073824 0.448959 0.032137            0
 기존지표 결측없음  재배치   570 308.000000        295 0.110683      0.073694 0.448390 0.032137            0
변화판정
동일    264
개선    236
악화    102
     읍면_key  현재반영_이동격차지수  재배치반영_이동격차지수  지수변화_재배치빼기현재
경기도 여주시 산북면     0.448959      

## 5. 결과 CSV·그림·보고서 저장

같은 파이프라인을 전체 실행하여 모든 산출물, 검증 기록과 README를 갱신합니다. 원본 inputs는 변경하지 않습니다.

In [6]:
result, weights, summary = analysis.main()
print('출력 파일 수:', len(list((base / 'outputs').glob('*'))))

        대상 시나리오  읍면동수       대수합계  공급양수_읍면동수     지수평균  지수표준편차_ddof0     지수최대     지수최소  기존지표결측_읍면동수
    경기도 전체   현재   602 326.000000        113 0.109971      0.072577 0.448959 0.032137           32
    경기도 전체  재배치   602 326.000000        313 0.109971      0.072437 0.448390 0.032137           32
기존6지표 상위20   현재    20   9.666667          7 0.348733      0.044480 0.448959 0.292746            0
기존6지표 상위20  재배치    20  33.000000         20 0.348070      0.044302 0.448390 0.292178            0
기존6지표 상위50   현재    50  31.000000         18 0.297420      0.052531 0.448959 0.231137            0
기존6지표 상위50  재배치    50  63.000000         50 0.297057      0.052252 0.448390 0.230569            0
 기존지표 결측없음   현재   570 286.000000         98 0.110704      0.073824 0.448959 0.032137            0
 기존지표 결측없음  재배치   570 308.000000        295 0.110683      0.073694 0.448390 0.032137            0

공통 가중치
          지표     엔트로피     정보효용      가중치
   지형장벽_취약점수 0.974112 0.025888 0.135261
 자가용미보유_취약점수 0.999929 0.000071

## 6. 시각화

![현재와 재배치 비교](figures/01_두시나리오_비교.png)

![공통 가중치](figures/02_공통가중치.png)

주요 결과: outputs/01_현재운행반영_이동격차지수.csv, outputs/02_재배치반영_이동격차지수.csv, outputs/03_두시나리오_전체비교.csv.

전체 방법·가정·한계·파일 안내는 [README](README.md)를 확인하세요.